In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
import gensim
from sklearn.model_selection import train_test_split

### Load Data

In [ ]:
df_data = pd.read_csv('./Data/data_processed.csv').dropna()
le = preprocessing.LabelEncoder()
df_data['class'] = le.fit_transform(df_data.genre)
df_data

In [ ]:
le.classes_

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_data[['clean_text', 'clean_title']],
                                                   df_data['class'], random_state = 42,
                                                   test_size = 0.2, stratify=df_data['class'])

### CounterVectorize

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer()
X_train_countvec = count_vect.fit_transform(list(X_train.clean_text)).toarray()
X_test_countvec = count_vect.transform(list(X_test.clean_text)).toarray()

In [ ]:
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB().fit(X_train_countvec, y_train)
y_pred = clf.predict(X_test_countvec)

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
print('Accuracy is {}'.format(accuracy_score(y_test, y_pred)))
confusion_matrix(y_test, y_pred)

In [ ]:
import numpy as np
import torch
from torch import nn
from torch import optim
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import Dataset,DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

In [ ]:
## NN model
class NN(nn.Module):
    def __init__(self,in_feature,out_feature):
        super().__init__()
        self.n1 = nn.Linear(in_feature,256)
        self.n2 = nn.Linear(256,128)
        self.n3 = nn.Linear(128,64)
        self.n4 = nn.Linear(64,out_feature)
    
    def forward(self, x):
        x = self.n1(x)
        x = F.relu(x)
        x = self.n2(x)
        x = F.relu(x)
        x = self.n3(x)
        x = F.relu(x)
        output = self.n4(x)
        return output
def loss_function(output,target):
    target_onehot = []
    for i in range(len(target)):
        temp = np.zeros(6)
        temp[target[i].int().cpu().detach().numpy()] = 1
        target_onehot.append(temp)
    tar_oh_np = np.array(target_onehot)
    tar_tensor = torch.tensor(tar_oh_np).to(device).float()
    return criterion(output,tar_tensor)

In [ ]:
def train(model,criterion, optimizer,loss_list1,loss_list2):
    train_loss = 0
    model.train()
    for data, target in custom_trainloader:
        data, target = data.to(device), target.to(device)
        data = data.float()
        target = target.float()
        optimizer.zero_grad()
        output = model(data)
        loss = loss_function(output,target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss = (train_loss)/len(custom_trainloader)
    loss_list1.append(train_loss)
    test_loss = 0
    model.eval()
    with torch.no_grad():
        for data, target in custom_valiloader:
            data, target = data.to(device), target.to(device)
            data = data.float()
            target = target.float()
            output = model(data)
            loss = loss_function(output,target)
            test_loss += loss.item() # sum up batch loss
    test_loss = (test_loss)/len(custom_valiloader)
    loss_list2.append(test_loss)
    print('Train({}): Loss: {:.4f}'.format(epoch, train_loss))


def plot_loss(losses_tr,losses_vali,upbound):
    plt.figure()
    plt.plot(list(range(len(losses_tr))), losses_tr, color='blue', linewidth=2, label='Train loss')
    plt.plot(list(range(len(losses_vali))), losses_vali, color='red', linewidth=2, label='Validation loss')
    plt.legend()
    plt.ylim(0,upbound)
    plt.xlabel('iterations')
    plt.ylabel('MSE')
    plt.grid()
    plt.show()

In [ ]:
class CustomDataset(torch.utils.data.Dataset):

    def __init__(self, X, Y, transform = None):
        self.X = X
        self.Y = Y
        self.transform = transform

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]
batch_size=256    
custom_trainset = CustomDataset(X_train_countvec,y_train.values)
custom_trainloader = torch.utils.data.DataLoader(custom_trainset, batch_size=batch_size, shuffle=True)
custom_valiset = CustomDataset(X_test_countvec,y_test.values)
custom_valiloader = torch.utils.data.DataLoader(custom_valiset, batch_size=batch_size, shuffle=True)

In [ ]:
## Training the model
seed = 4
device = 'cuda'
num_epochs = 20


device = torch.device(device)
torch.manual_seed(1)
torch.cuda.manual_seed(1)
model = NN(35781,6).to(device)
optimizer = optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-8)
criterion = nn.MSELoss()
loss_tr1 = []
loss_te1 = []
for epoch in range(1, num_epochs + 1):
    train(model, criterion, optimizer,loss_tr1,loss_te1)
    #test(model, device, custom_valiloader, criterion, epoch,loss_te1)


loss_tr = [] 
loss_te = []
for i in range(len(loss_tr1)):
    loss_tr.append(loss_tr1[i])
for i in range(len(loss_te1)):
    loss_te.append(loss_te1[i])

In [ ]:
plot_loss(loss_tr,loss_te,.5)

In [ ]:

from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred,digits=4))


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
cm = confusion_matrix(y_test,y_pred)

#??confusion matrix
print("Confusion Matrix")
category_labels = ['crime', 'fantasy', 'history', 'horror', 'science', 'thriller']
cm_normalised = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.set(font_scale=1.5)
fig, ax = plt.subplots(figsize=(10,10))
ax = sns.heatmap(cm_normalised, annot=True, linewidths=1, square=False,
                cmap="Greens", yticklabels=category_labels, xticklabels=category_labels, vmin=0, vmax=np.max(cm_normalised),
                fmt=".2f", annot_kws={"size": 20})
ax.set(xlabel='Predicted label', ylabel='True label')

### TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vector = TfidfVectorizer()
X_train_tfidf = tfidf_vector.fit_transform(list(X_train.clean_text))
X_test_tfidf = tfidf_vector.transform(list(X_test.clean_text))

In [ ]:
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB().fit(X_train_tfidf, y_train)
y_pred = clf.predict(X_test_tfidf)

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
print('Accuracy is {}'.format(accuracy_score(y_test, y_pred)))
confusion_matrix(y_test, y_pred)

In [ ]:
class NN(nn.Module):
    def __init__(self,in_feature,out_feature):
        super().__init__()
        self.n1 = nn.Linear(in_feature,1024)
        self.n2 = nn.Linear(1024,256)
        self.n3 = nn.Linear(256,64)
        self.n4 = nn.Linear(64,out_feature)
    
    def forward(self, x):
        x = self.n1(x)
        x = F.relu(x)
        x = self.n2(x)
        x = F.relu(x)
        x = self.n3(x)
        x = F.relu(x)
        output = self.n4(x)
        return output

In [ ]:
batch_size=200
custom_trainset = CustomDataset(X_train_tfidf.toarray(),y_train.values)
custom_trainloader = torch.utils.data.DataLoader(custom_trainset, batch_size=batch_size, shuffle=True)
custom_valiset = CustomDataset(X_test_tfidf.toarray(),y_test.values)
custom_valiloader = torch.utils.data.DataLoader(custom_valiset, batch_size=batch_size, shuffle=True)

In [ ]:
seed = 1
device = 'cuda'
num_epochs = 20


device = torch.device(device)
torch.manual_seed(1)
torch.cuda.manual_seed(1)
model = NN(35781,6).to(device)
optimizer = optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-8)
criterion = nn.MSELoss()
loss_tr1 = []
loss_te1 = []
for epoch in range(1, num_epochs + 1):
    train(model, criterion, optimizer,loss_tr1,loss_te1)
    #test(model, device, custom_valiloader, criterion, epoch,loss_te1)


loss_tr = [] 
loss_te = []
for i in range(len(loss_tr1)):
    loss_tr.append(loss_tr1[i])
for i in range(len(loss_te1)):
    loss_te.append(loss_te1[i])

In [ ]:
plot_loss(loss_tr,loss_te,.5)

In [ ]:
print(classification_report(y_test,y_pred,digits=4))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
cm = confusion_matrix(y_test,y_pred)

#??confusion matrix
print("Confusion Matrix")
category_labels = ['crime', 'fantasy', 'history', 'horror', 'science', 'thriller']
cm_normalised = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.set(font_scale=1.5)
fig, ax = plt.subplots(figsize=(10,10))
ax = sns.heatmap(cm_normalised, annot=True, linewidths=1, square=False,
                cmap="Greens", yticklabels=category_labels, xticklabels=category_labels, vmin=0, vmax=np.max(cm_normalised),
                fmt=".2f", annot_kws={"size": 20})
ax.set(xlabel='Predicted label', ylabel='True label')

### Word2Vec

In [ ]:
import gensim
from nltk import word_tokenize, sent_tokenize

sent_words = [word_tokenize(w) for w in list(X_train.clean_text)]
model = gensim.models.Word2Vec(sent_words, sg=1, vector_size =200, window=3,epochs=5,
                               min_count=3, negative=3, sample=0.001, hs=1)
model.wv.save_word2vec_format('./word2vec_model.txt', binary=False)

In [ ]:
def average_word_vectors(text,model,num_features):
    vocabulary=set(w2v_model.index_to_key)
    words = word_tokenize(text)
    feature_vector=np.zeros((num_features,),dtype='float64')
    nwords=0
    for word in words:
        if word in vocabulary:
            nwords=nwords+1
            feature_vector=np.add(feature_vector,model[word])
    if nwords:
        feature_vector=np.divide(feature_vector,nwords)
    return feature_vector


def text2vector(df, w2v_model, num_features):
    # prepare vectors for train/tex=st data
    vectors = []
    for i in range(len(df)):
        val = average_word_vectors(df['clean_text'].iloc[i], w2v_model, num_features)
        vectors.append(val)
    return np.array(vectors)


w2v_model = gensim.models.KeyedVectors.load_word2vec_format('word2vec_model.txt',binary=False)
num_features = 200

In [ ]:
from sklearn import svm

clf = svm.SVC().fit(text2vector(X_train, w2v_model, num_features), y_train)
y_pred = clf.predict(text2vector(X_test, w2v_model, num_features))

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
print('Accuracy is {}'.format(accuracy_score(y_test, y_pred)))
confusion_matrix(y_test, y_pred)

In [ ]:
class NN(nn.Module):
    def __init__(self,in_feature,out_feature):
        super().__init__()
        self.n1 = nn.Linear(in_feature,64)
        self.n2 = nn.Linear(64,16)
        self.n3 = nn.Linear(16,out_feature)
    
    def forward(self, x):
        x = self.n1(x)
        x = F.relu(x)
        x = self.n2(x)
        x = F.relu(x)
        output = self.n3(x)
        return output

In [ ]:
batch_size=512
custom_trainset = CustomDataset(text2vector(X_train, w2v_model, num_features),y_train.values)
custom_trainloader = torch.utils.data.DataLoader(custom_trainset, batch_size=batch_size, shuffle=True)
custom_valiset = CustomDataset(text2vector(X_test, w2v_model, num_features),y_test.values)
custom_valiloader = torch.utils.data.DataLoader(custom_valiset, batch_size=batch_size, shuffle=True)

In [ ]:
seed = 1
device = 'cuda'
num_epochs = 100


device = torch.device(device)
torch.manual_seed(1)
torch.cuda.manual_seed(1)
model = NN(200,6).to(device)
optimizer = optim.Adam(model.parameters(),lr=0.01,weight_decay=1e-8)
criterion = nn.MSELoss()
loss_tr1 = []
loss_te1 = []
for epoch in range(1, num_epochs + 1):
    train(model, criterion, optimizer,loss_tr1,loss_te1)
    #test(model, device, custom_valiloader, criterion, epoch,loss_te1)


loss_tr = [] 
loss_te = []
for i in range(len(loss_tr1)):
    loss_tr.append(loss_tr1[i])
for i in range(len(loss_te1)):
    loss_te.append(loss_te1[i])

In [ ]:
plot_loss(loss_tr,loss_te,.5)

In [ ]:
print(classification_report(y_test,y_pred,digits=4))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
cm = confusion_matrix(y_test,y_pred)

#??confusion matrix
print("Confusion Matrix")
category_labels = ['crime', 'fantasy', 'history', 'horror', 'science', 'thriller']
cm_normalised = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.set(font_scale=1.5)
fig, ax = plt.subplots(figsize=(10,10))
ax = sns.heatmap(cm_normalised, annot=True, linewidths=1, square=False,
                cmap="Greens", yticklabels=category_labels, xticklabels=category_labels, vmin=0, vmax=np.max(cm_normalised),
                fmt=".2f", annot_kws={"size": 20})
ax.set(xlabel='Predicted label', ylabel='True label')